In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install mlflow dagshub

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import mlflow.sklearn
from sklearn.base import BaseEstimator, TransformerMixin

os.environ['MLFLOW_TRACKING_USERNAME'] = 'aleko-mamukashvili'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '02f60ba77ab2caa591fcc8dfffd1a9744cbf2694'

class MasterUIDTransformer(BaseEstimator, TransformerMixin):
    def __init__(self): pass
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        if 'card1' in X.columns and 'card2' in X.columns:
            X['uid'] = X['card1'].astype(str) + '_' + X['card2'].astype(str)
            if 'addr1' in X.columns:
                X['uid2'] = X['uid'] + '_' + X['addr1'].astype(str)
        return X

sys.path.insert(0, os.getcwd())
try:
    from my_preprocessing_classes import FraudDataCleaner, FraudFeatureEngineer, FraudEncoder, reduce_mem_usage
except ImportError:
    def reduce_mem_usage(df):
        for col in df.columns:
            if df[col].dtype != object:
                c_min, c_max = df[col].min(), df[col].max()
                if str(df[col].dtype)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    else:
                        df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.float32)
        return df

print("მოდელის ჩატვირთვა...")
mlflow.set_tracking_uri("https://dagshub.com/aleko-mamukashvili/-IEEE-CIS-Fraud-Detection..mlflow")
model = mlflow.sklearn.load_model("models:/final/1")

print("მონაცემების ჩატვირთვა...")
base_path = '/kaggle/input/competitions/ieee-fraud-detection/'
test_raw = reduce_mem_usage(pd.read_csv(f'{base_path}test_transaction.csv'))

print("პროგნოზირება...")
try:
    test_probs = model.predict_proba(test_raw)[:, 1]
except ValueError as e:
    print(f"ერორი: {e}")
    X_input = test_raw.drop(columns=['TransactionID'])
    test_probs = model.predict_proba(X_input)[:, 1]

print("ფაილის შენახვა...")
sample_sub = pd.read_csv(f'{base_path}sample_submission.csv')
submission = pd.DataFrame({'TransactionID': test_raw['TransactionID'], 'isFraud': test_probs})
final_sub = sample_sub[['TransactionID']].merge(submission, on='TransactionID', how='left')
final_sub['isFraud'] = final_sub['isFraud'].fillna(0.02)
final_sub.to_csv('/kaggle/working/submission.csv', index=False)
print("✅ წარმატება!")

In [ ]:
import os
print(os.listdir('/kaggle/working/'))